Run once with **Accelerator: None**, Internet on. Other notebooks attach this output instead of downloading again.

In [ ]:
from pathlib import Path

OUT = Path("/kaggle/working")
(OUT / "ckpt").mkdir(exist_ok=True)
(OUT / "data").mkdir(exist_ok=True)

In [ ]:
from huggingface_hub import hf_hub_download

for repo, filename in [
    ("nieshen/SMDM", "mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors"),
    ("taeyoun811/whisfusion", "whisfusion_stage2_decoder.pt"),
]:
    print(hf_hub_download(repo, filename, local_dir=str(OUT / "ckpt")))

In [ ]:
import subprocess

# test-clean to measure on, dev-* to tune on
for split in ["test-clean", "dev-clean", "dev-other"]:
    target = OUT / "data" / "LibriSpeech" / split
    if target.exists():
        continue

    archive = OUT / f"{split}.tar.gz"
    subprocess.run(["curl", "-L", "--retry", "3", "-o", str(archive),
                    f"https://www.openslr.org/resources/12/{split}.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(OUT / "data")], check=True)
    archive.unlink()

    print(split, len(list(target.rglob("*.flac"))))

In [ ]:
from transformers import AutoTokenizer, WhisperForConditionalGeneration, WhisperProcessor

CACHE = str(OUT / "hf")
WhisperProcessor.from_pretrained("openai/whisper-small", cache_dir=CACHE)
WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", cache_dir=CACHE)
AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
                              cache_dir=CACHE)